In [1]:
import pandas as pd
import numpy as np

## SMS Spam Classification model

In [2]:
df = pd.read_csv('/content/drive/MyDrive/ML_Datasets/SMSSpamCollection.csv')
df.head()

,label,sms_message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
df['sms_message'][120]

'PRIVATE! Your 2004 Account Statement for 07742676969 shows 786 unredeemed Bonus Points. To claim call 08719180248 Identifier Code: 45239 Expires'

In [6]:
df['label'].value_counts()/len(df)
# Little class imbalance is there as spam is in minority

,count
label,
ham,0.865937
spam,0.134063


In [7]:
# Encode target variable
label_map = {'ham':1,'spam':0}
df['target'] = df['label'].apply(lambda x:label_map[x])
df

,label,sms_message,target
0,ham,"Go until jurong point, crazy.. Available only ...",1
1,ham,Ok lar... Joking wif u oni...,1
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,0
3,ham,U dun say so early hor... U c already then say...,1
4,ham,"Nah I don't think he goes to usf, he lives aro...",1
...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,0
5568,ham,Will ü b going to esplanade fr home?,1
5569,ham,"Pity, * was in mood for that. So...any other s...",1
5570,ham,The guy did some bitching but I acted like i'd...,1


## Text Preprocessing

In [15]:
import re #text cleaning
import nltk # NLP library
from nltk.tokenize import word_tokenize #tokenization
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer # stemming

from nltk.stem import WordNetLemmatizer

nltk.download('punkt_tab')
nltk.download('stopwords')


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [13]:
# stemmer objet
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

In [31]:
def preprocess(raw_text,stem_or_lemm='stem'):
  # Remove special chars from text using regex
  sent = re.sub(r'[^a-zA-Z0-9 ]',' ',raw_text)
  #eg. "Hello, World 0304 ??" -> "Hello  World 0304  "

  #Lowercase the sent
  sent=sent.lower()

  #tokenize into words
  tokens = word_tokenize(sent)
  #"hello  world 0304  " -> ['hello', 'world', '0304']

  #Remove stopwords
  clean_tokens = [t for t in tokens if not t in stopwords.words('english')]

  # Stemming / Lemmatization

  if stem_or_lemm =='stem':
    clean_tokens = [stemmer.stem(t) for t in clean_tokens]
  else:
    clean_tokens = [lemmatizer.lemmatize(t) for t in clean_tokens]

  return pd.Series([' '.join(clean_tokens)])

In [32]:
df['processed_text']=df['sms_message'].apply(lambda x:preprocess(x))
df

,label,sms_message,target,processed_text
0,ham,"Go until jurong point, crazy.. Available only ...",1,go jurong point crazi avail bugi n great world...
1,ham,Ok lar... Joking wif u oni...,1,ok lar joke wif u oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,0,free entri 2 wkli comp win fa cup final tkt 21...
3,ham,U dun say so early hor... U c already then say...,1,u dun say earli hor u c alreadi say
4,ham,"Nah I don't think he goes to usf, he lives aro...",1,nah think goe usf live around though
...,...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,0,2nd time tri 2 contact u u 750 pound prize 2 c...
5568,ham,Will ü b going to esplanade fr home?,1,b go esplanad fr home
5569,ham,"Pity, * was in mood for that. So...any other s...",1,piti mood suggest
5570,ham,The guy did some bitching but I acted like i'd...,1,guy bitch act like interest buy someth els nex...


In [35]:
## train test split
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(df['processed_text'],df['target'],test_size=0.2)

print("Total datapoints = ",len(df))
print("Train datapoints = ",len(x_train))
print("Test datapoints = ",len(x_test))

Total datapoints =  5572
Train datapoints =  4457
Test datapoints =  1115


In [36]:
x_train

,processed_text
3519,will go app class
5534,ok anoth number
1038,naughti littl thought better flirt flirt n fli...
1836,septemb
171,sir need axi bank account bank address
...,...
4027,oh ok wat ur email
1242,want show world princess europ
2832,thanx 4 send home
343,u hide stranger


In [39]:
# Buid Frequency Table for this data
from sklearn.feature_extraction.text import CountVectorizer
count_vectorizer = CountVectorizer()

train_data = count_vectorizer.fit_transform(x_train)
train_data

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 38114 stored elements and shape (4457, 6471)>

In [43]:
vocab = count_vectorizer.get_feature_names_out()
print("Total vocab size = ",len(vocab))

Total vocab size =  6471


In [42]:
vocab[1000:1020]

array(['arngd', 'arnt', 'around', 'arr', 'arrang', 'arrest', 'arriv',
       'arrow', 'arsen', 'art', 'arti', 'artist', 'arul', 'arun', 'asa',
       'asap', 'asda', 'ashley', 'ashwini', 'asia'], dtype=object)

In [45]:
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()

nb_model.fit(train_data,y_train)

MultinomialNB()

In [46]:
y_pred = nb_model.predict(train_data)

In [47]:
y_pred

array([1, 1, 1, ..., 1, 1, 1])

In [49]:
from sklearn.metrics import classification_report

print(classification_report(y_train,y_pred))

              precision    recall  f1-score   support

           0       0.97      0.98      0.97       607
           1       1.00      1.00      1.00      3850

    accuracy                           0.99      4457
   macro avg       0.98      0.99      0.99      4457
weighted avg       0.99      0.99      0.99      4457



In [50]:
test_data = count_vectorizer.transform(x_test)
test_data

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 8233 stored elements and shape (1115, 6471)>

In [51]:
y_pred = nb_model.predict(test_data)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.98      0.91      0.94       140
           1       0.99      1.00      0.99       975

    accuracy                           0.99      1115
   macro avg       0.99      0.95      0.97      1115
weighted avg       0.99      0.99      0.99      1115



In [ ]:
## Conclusion : Model is able to give 90%+ in both train and testing
## No overfitting or underfiting observed